<a href="https://colab.research.google.com/github/ahmedmoustafa/datasets/blob/main/happiness/happiness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# World Happiness: Annual Data, 2005–2023

Install pandas before running this setup. Add a plotting library for your answers. Read the [dataset notes](https://github.com/ahmedmoustafa/datasets/blob/main/happiness/README.md). Downloads are pinned to a fixed version. The original and processed tables serve different purposes; this notebook shows their differences before analysis.

In [ ]:
import pandas as pd

base_url = "https://media.githubusercontent.com/media/ahmedmoustafa/datasets/4f2478a00774897b3a3cfce99466bc97a620eb5a/happiness/"
expected_columns = ['country', 'year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'healthy_life_expectancy_at_birth', 'freedom_to_make_life_choices', 'generosity', 'perceptions_of_corruption', 'positive_affect', 'negative_affect']
df_original = pd.read_csv(base_url + "happiness_original.tsv", sep="\t")
df_happiness = pd.read_csv(base_url + "happiness.tsv", sep="\t")
df_countries = pd.read_csv(base_url + "countries.tsv", sep="\t")
for table, rows in [(df_original, 2363), (df_happiness, 2362)]:
    assert list(table.columns) == expected_columns
    assert table.shape == (rows, 11)
    assert not table[["country", "year"]].isna().any().any()
    assert not table.duplicated(["country", "year"]).any()
assert list(df_countries.columns) == ["country", "continent", "region", "economic_status", "political_system"]
assert df_countries.shape == (191, 5)
assert df_countries["country"].notna().all() and df_countries["country"].is_unique
print("Loaded original, processed, and lookup tables.")

## Check the local preprocessing

The original imputation program is unavailable. This comparison reproduces the audit, not the random-forest estimates. Use `df_original` for analyses that should exclude locally imputed values, or use `imputed_mask` to flag them in the processed table. The omitted Egypt–2005 record remains available in `excluded_records` and `df_original`.

In [ ]:
keys = ["country", "year"]
original = df_original.set_index(keys)
processed = df_happiness.set_index(keys)
assert processed.index.isin(original.index).all()
aligned_original = original.reindex(processed.index)

# Flags identify values filled locally, not all upstream estimation.
imputed_mask = aligned_original.isna() & processed.notna()
observed = aligned_original.notna()
assert not (observed & processed.ne(aligned_original)).any().any()
assert not (observed & processed.isna()).any().any()
imputed_counts = imputed_mask.sum()
excluded_records = df_original.loc[~original.index.isin(processed.index)].copy()
assert int(imputed_counts.sum()) == 384
assert excluded_records[keys].to_dict("records") == [{"country": "Egypt", "year": 2005}]
print("Locally filled cells by variable:")
print(imputed_counts[imputed_counts > 0])
print("Excluded country-years:")
print(excluded_records[keys].to_string(index=False))

## Join without dropping country-years

The lookup has no documented source or reference year. The explicit aliases below only match names; they do not validate geographic, economic, or political labels. Unmatched countries remain in the result with missing metadata.

In [ ]:
# Explicit spelling aliases only; preserve original country labels.
country_aliases = {'Congo (Brazzaville)': 'Congo, Republic of the', 'Congo (Kinshasa)': 'Congo, Democratic Republic of the', 'Czechia': 'Czech Republic', 'South Korea': 'Korea, South', 'Taiwan Province of China': 'Taiwan', 'Türkiye': 'Turkey'}
lookup = df_countries.rename(columns={"country": "lookup_country"})
with_keys = df_happiness.assign(
    lookup_country=df_happiness["country"].replace(country_aliases)
)
df_with_metadata = with_keys.merge(
    lookup, on="lookup_country", how="left", validate="many_to_one", indicator=True
)
assert len(df_with_metadata) == len(df_happiness)
unmatched = df_with_metadata.loc[df_with_metadata["_merge"] == "left_only", ["country", "year"]]
print("Rows retained:", len(df_with_metadata))
print("Rows without lookup metadata:", len(unmatched))
print(unmatched.groupby("country").size().to_string())
# Do not fill missing metadata with another country's classifications.


## Q1. For a selected year, summarize life_ladder across countries. State whether the summary weights each country equally; it is not a population-weighted average.

## Q2. For a selected year, compare life_ladder across available lookup regions. Report unmatched countries separately and state the lookup limitations.

## Q3. What is the range of `log_gdp_per_capita` values, and which countries have the highest and lowest GDP per capita?

## Q4. How has the `life_ladder` score changed over time in a specific country?

## Q5. Choose common start and end years. Which countries have the largest change in life_ladder? Include only countries observed in both years and report exclusions.

## Q6. Are there noticeable trends in `healthy_life_expectancy_at_birth` over time across regions or continents?

## Q7. What is the relationship between `log_gdp_per_capita` and `life_ladder`? Do countries with higher GDP per capita tend to have higher happiness scores?

## Q8. How does `social_support` correlate with `life_ladder`? Do countries with higher social support report higher happiness?

## Q9. Is there a relationship between `freedom_to_make_life_choices` and `life_ladder`?

## Q10. How do `positive_affect` and `negative_affect` differ across countries? Are there countries with high positive affect and low negative affect, or vice versa?

## Q11. Is there a correlation between `life_ladder` and `positive_affect`? What about `life_ladder` and `negative_affect`?

## Q12. What association appears between perceptions_of_corruption and life_ladder? Explain why this comparison does not establish a causal effect.

## Q13. How does the `generosity` measure correlate with `log_gdp_per_capita`? Are wealthier countries generally more or less generous?

## Q14. How do different regions or continents compare in terms of `life_ladder`, `social_support`, and `freedom_to_make_life_choices`?

## Q15. Explore healthy life expectancy by the recorded economic labels. Explain why the undated lookup cannot establish historical economic status.

## Q16. Create a time-series chart showing `life_ladder` over time for a selected group of countries.

## Q17. Use scatter plots to explore relationships between `log_gdp_per_capita` and `life_ladder`, color-coded by continent or region.

## Q18. Compare an analysis using original observed values with one using the processed table. Report which variables and country-years were imputed, the excluded record, and whether your conclusion changes.